In [1]:
import yaml
import os
import re
import pandas as pd
from typing import List, Dict, Tuple, Any

# Define the accessibility variable patterns
# Tuple format: (regex_pattern, category_name)
ACCESS_PATTERNS: List[Tuple[str, str]] = [
    (r'nodes_walk_', 'nodes_walk'),
    (r'zones_transit_jobs_', 'transit_jobs'),
    (r'jobs_within_\d+_min', 'travel_time_jobs'),
    (r'zones_', 'zones_any'),
    (r'logsum_', 'logsum'),
    (r'parcels_zones_', 'parcels_zones_bridge'),
    # Added general accessibility terms based on sample provided for robustness
    (r'walk_nearest_', 'walk_nearest_facility'), 
    (r'nodes_drv_log_sum_', 'nodes_drv_log_sum'),
]

def find_accessibility_variables(variable_name: str) -> str:
    """
    Checks if a variable name matches any accessibility pattern.
    Returns the category name if a match is found, otherwise returns None.
    Handles interaction terms (separated by ':') by checking all parts.
    """
    # Split by ':' to handle interaction terms (e.g., 'high_income:zones_logsum_...')
    parts = variable_name.split(':')
    for pattern, category in ACCESS_PATTERNS:
        # Check if the pattern matches any part of the variable name
        if any(re.search(pattern, part) for part in parts):
            return category
    return None

def extract_model_summary(model_name: str, yaml_data: Dict[str, Any]) -> pd.DataFrame:
    """
    Parses the summary_table string in the YAML data to extract coefficients, 
    p-values, and variable names, then identifies accessibility variables.
    """
    summary_table_str = yaml_data.get('saved_object', {}).get('summary_table', '')
    model_expression_str = yaml_data.get('saved_object', {}).get('model_expression', '')

    # --- 1. Parse the Summary Table ---
    
    start_marker = "-----------------------------------------------------------------------------------------------------"
    end_marker = "====================================================================================================="
    
    try:
        start_index = summary_table_str.rfind(start_marker)
        end_index = summary_table_str.rfind(end_marker)
        
        if start_index == -1 or end_index == -1:
            print(f"Warning: Summary table markers not found for model '{model_name}'. Skipping data extraction.")
            return pd.DataFrame() 

        # Extract the relevant lines, skipping the markers
        table_lines = summary_table_str[start_index + len(start_marker):end_index].strip().split('\n')
        
        # Determine variable names from the model expression, in order
        variable_names = [
            var.strip()
            # Removes the constant/intercept term (' - 1') and splits by ' + '
            for var in model_expression_str.replace(' - 1', '').split(' + ') 
            if var.strip()
        ]
        
        # Regex to capture the numerical values (coef, std err, z, P>|z|)
        # Looks for four floating-point numbers at the end of the line
        data_pattern = re.compile(r'\s+([-+]?\d+\.\d+)\s+([-+]?\d+\.\d+)\s+([-+]?\d+\.\d+)\s+([-+]?\d+\.\d+)')
        
        extracted_data = []
        for line in table_lines:
            match = data_pattern.search(line)
            if match:
                # The groups are: coef, std err, z_value, p_value
                extracted_data.append([float(x) for x in match.groups()])

        if not variable_names or len(variable_names) != len(extracted_data):
            print(f"Warning: Mismatch between extracted variables ({len(variable_names)}) and data rows ({len(extracted_data)}) for model '{model_name}'. Skipping.")
            return pd.DataFrame()

        # Create DataFrame
        df = pd.DataFrame(extracted_data, 
                          columns=['coefficient', 'std_err', 'z_value', 'p_value'])
        df['variable'] = variable_names
        df['model_name'] = model_name

    except Exception as e:
        print(f"Error processing summary table for model '{model_name}': {e}")
        return pd.DataFrame()

    # --- 2. Identify Accessibility Variables and Category ---
    df['is_accessibility'] = df['variable'].apply(lambda x: find_accessibility_variables(x) is not None)
    df['accessibility_category'] = df['variable'].apply(find_accessibility_variables)
    
    # Filter only the accessibility variables for the final output
    df_accessibility = df[df['is_accessibility']].copy()
    
    return df_accessibility[['model_name', 'variable', 'accessibility_category', 
                             'coefficient', 'p_value']]

def analyze_accessibility_impact(yaml_dir_path: str, file_list: List[str]):
    """
    Main function to load YAML files, extract data, and perform analysis.
    """
    all_accessibility_data = []
    
    print(f"--- 📂 Loading and Parsing Models from: {yaml_dir_path} ---")

    for file_name in file_list:
        file_path = os.path.join(yaml_dir_path, file_name)
        model_name = os.path.splitext(file_name)[0]
        
        if not os.path.exists(file_path):
            print(f"❌ Error: File not found at {file_path}. Skipping.")
            continue

        try:
            with open(file_path, 'r') as f:
                yaml_data = yaml.safe_load(f)
            
            df_model = extract_model_summary(model_name, yaml_data)
            if not df_model.empty:
                all_accessibility_data.append(df_model)
                print(f"✅ Successfully processed model: {model_name}")
            else:
                print(f"⚠️ Processed model: {model_name}, but found no valid accessibility data.")

        except yaml.YAMLError as e:
            print(f"❌ Error: Could not parse YAML file {file_name}. Details: {e}")
        except Exception as e:
            print(f"❌ An unexpected error occurred while processing {file_name}: {e}")

    if not all_accessibility_data:
        print("\n--- 🛑 No valid accessibility data was found for analysis. ---")
        return

    # Combine all dataframes
    combined_df = pd.concat(all_accessibility_data, ignore_index=True)

    # --- 1. Summary of Accessibility Variables Used per Model ---
    
    print("\n" + "="*80)
    print("## 📊 Summary of Accessibility Variables Used per Model")
    print("="*80)
    
    # Count variables and categories per model
    model_summary = combined_df.groupby('model_name').agg(
        total_accessibility_variables=('variable', 'size'),
        unique_categories=('accessibility_category', 'nunique')
    ).reset_index()
    
    print("\n### Total Accessibility Variables and Categories per Model")
    print(model_summary.to_markdown(index=False))

    # --- 2. P-Value Significance Analysis ---
    
    # p-value < 0.05 is typically considered statistically significant.
    SIGNIFICANCE_THRESHOLD = 0.05
    
    significant_vars = combined_df[combined_df['p_value'] < SIGNIFICANCE_THRESHOLD]
    
    print("\n" + "="*80)
    print(f"## ⭐ Significance Analysis (P-Value < {SIGNIFICANCE_THRESHOLD})")
    print("="*80)
    
    # Group by model to show the count of significant variables
    sig_summary_per_model = significant_vars.groupby('model_name').size().reset_index(name='significant_count')
    sig_summary_per_model = model_summary.merge(sig_summary_per_model, on='model_name', how='left').fillna(0)
    sig_summary_per_model['significant_count'] = sig_summary_per_model['significant_count'].astype(int)
    sig_summary_per_model['%_significant'] = (sig_summary_per_model['significant_count'] / sig_summary_per_model['total_accessibility_variables'] * 100).round(1)

    print("\n### Count and Percentage of Significant Accessibility Variables per Model")
    print(sig_summary_per_model[['model_name', 'total_accessibility_variables', 'significant_count', '%_significant']].to_markdown(index=False))
    
    # Summary of significant variables by category (across all models)
    significant_by_category = significant_vars.groupby('accessibility_category').size().sort_values(ascending=False).reset_index(name='count_significant')
    print("\n### Significant Accessibility Variables by Category (Total Count)")
    print(significant_by_category.to_markdown(index=False))

    # --- 3. Top 10 Most Impactful Accessibility Variables (by absolute coefficient) ---
    
    print("\n" + "="*80)
    print("## 🏆 Top 10 Most Impactful Accessibility Variables (Absolute Coefficient)")
    print("="*80)
    
    # Create a column for absolute coefficient for sorting
    combined_df['abs_coefficient'] = combined_df['coefficient'].abs()
    
    # Sort by absolute coefficient (descending) and get the top 10 unique variables
    # Dropping duplicates based on 'variable' ensures we only see each unique variable once, 
    # taking the entry from the model with the highest absolute coefficient (which is the first one after sorting).
    top_10_impact = combined_df.sort_values(by='abs_coefficient', ascending=False).drop_duplicates(subset=['variable']).head(30)
    
    # Prepare the final output table for the top 10
    top_10_impact_output = top_10_impact[['variable', 'model_name', 'accessibility_category', 'coefficient', 'abs_coefficient', 'p_value']]
    
    # Format the numerical columns for clean display
    top_10_impact_output['coefficient'] = top_10_impact_output['coefficient'].map('{:,.4f}'.format)
    top_10_impact_output['abs_coefficient'] = top_10_impact_output['abs_coefficient'].map('{:,.4f}'.format)
    top_10_impact_output['p_value'] = top_10_impact_output['p_value'].map('{:.3f}'.format)
    
    print("\n### Ranking of Variables by Magniture of Impact (Absolute Value of Coefficient)")
    print(top_10_impact_output.to_markdown(index=False))

In [4]:
# 1. Define the directory path based on your `ll` output
YAML_DIRECTORY = '/mnt/semcog_urbansim/configs/hlcm_2050' 

# 2. List the YAML files to be analyzed, based on your `ll` output
# This assumes the files listed are the complete set you want to analyze.
YAML_FILES_TO_ANALYZE = [
    'hlcm_115.yaml',
    'hlcm_125.yaml',
    'hlcm_147.yaml',
    'hlcm_161.yaml',
    'hlcm_3.yaml',
    'hlcm_5.yaml',
    'hlcm_93.yaml',
    'hlcm_99.yaml',
]

In [5]:
analyze_accessibility_impact(YAML_DIRECTORY, YAML_FILES_TO_ANALYZE)

--- 📂 Loading and Parsing Models from: /mnt/semcog_urbansim/configs/hlcm_2050 ---
✅ Successfully processed model: hlcm_115
✅ Successfully processed model: hlcm_125
✅ Successfully processed model: hlcm_147
✅ Successfully processed model: hlcm_161
✅ Successfully processed model: hlcm_3
✅ Successfully processed model: hlcm_5
✅ Successfully processed model: hlcm_93
✅ Successfully processed model: hlcm_99

## 📊 Summary of Accessibility Variables Used per Model

### Total Accessibility Variables and Categories per Model
| model_name   |   total_accessibility_variables |   unique_categories |
|:-------------|--------------------------------:|--------------------:|
| hlcm_115     |                              37 |                   5 |
| hlcm_125     |                              32 |                   4 |
| hlcm_147     |                              26 |                   4 |
| hlcm_161     |                              41 |                   5 |
| hlcm_3       |                          

In [ ]:
import pandas as pd
import io

# Data provided by the user
data = """
| model_name   |   total_accessibility_variables |   significant_count |   %_significant |
|:-------------|--------------------------------:|--------------------:|----------------:|
| Monroe     |                              37 |                  18 |            48.6 |
| Oakland     |                              32 |                  13 |            40.6 |
| St. Clair     |                              26 |                  15 |            57.7 |
| Washtenaw     |                              41 |                  33 |            80.5 |
| Wayne(exclude Detroit)       |                              32 |                  24 |            75.0 |
| Detroit       |                              31 |                  21 |            67.7 |
| Livingston      |                              31 |                  19 |            61.3 |
| Macomb      |                              29 |                  14 |            48.3 |
"""

# Read the data from the string, skipping the header and separator lines
df_raw = pd.read_csv(io.StringIO(data), sep='|', skiprows=[2], skipinitialspace=True).dropna(axis=1, how='all').iloc[1:]
df_raw.columns = [col.strip() for col in df_raw.columns]

# Convert columns to appropriate types
df_raw['total_accessibility_variables'] = df_raw['total_accessibility_variables'].astype(int)
df_raw['significant_count'] = df_raw['significant_count'].astype(int)
df_raw['%_significant'] = df_raw['%_significant'].astype(float)

# Calculate non-significant count
df_raw['non_significant_count'] = df_raw['total_accessibility_variables'] - df_raw['significant_count']

# Prepare data for stacking (long format)
df_significant = df_raw[['model_name', 'significant_count']].copy()
df_significant.rename(columns={'significant_count': 'count'}, inplace=True)
df_significant['Significance'] = 'Significant (P<0.05)'

df_not_significant = df_raw[['model_name', 'non_significant_count']].copy()
df_not_significant.rename(columns={'non_significant_count': 'count'}, inplace=True)
df_not_significant['Significance'] = 'Not Significant (P≥0.05)'

df_long = pd.concat([df_significant, df_not_significant])

# Add a sorting column based on the number of significant variables
sort_order = df_raw.sort_values('significant_count', ascending=False)['model_name'].tolist()

# Create the stacked bar chart
chart = alt.Chart(df_long).mark_bar().encode(
    # Sort X-axis by the significant count
    x=alt.X('model_name:N', sort=sort_order, title='Model Name'),
    y=alt.Y('count:Q', title='Number of Accessibility Variables'),
    color=alt.Color('Significance:N', 
                    title='Statistical Significance',
                    scale=alt.Scale(domain=['Significant (P<0.05)', 'Not Significant (P≥0.05)'],
                                    range=['#1f77b4', '#ff7f0e'])), # Blue for Significant, Orange for Not
    order=alt.Order('Significance', sort='descending'), # Ensure 'Significant' is on the bottom/base
    tooltip=['model_name', 'Significance', 'count']
).properties(
    title='Total Accessibility Variables by Significance (Stacked Bar Chart)'
).save('stacked_significance_bar_chart.json')

ImportError: cannot import name 'soft_unicode' from 'markupsafe' (/opt/conda/envs/forecast/lib/python3.9/site-packages/markupsafe/__init__.py)